# 02 — Classification Dev
### Part 2: Taxonomy & Classification + Attribute Extraction

Dev/scratch notebook for iterating on `part2_classification` (classify.py,
attributes.py, dedup.py) against the 200-item ground truth.

**Status:** real data files (200-item ground truth, `Unicat_Lov`, `FAUCETS_LOV`,
manufacturer/brand list) aren't in `data/raw/` yet, so this notebook runs
against a small **mock Faucets fixture** so the pipeline is exercisable today.

**When real files land:** only Section 2 (data loading) changes — swap the
mock loader for calls into `part1_foundation`'s real loaders. Sections 3
onward (classify → extract attributes → score → dedup) should run unchanged.

## 1. Setup

In [1]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

from unihack.part2_classification import (
    classify_batch, score_classification_accuracy,
    extract_attributes_batch, score_attribute_extraction,
    find_duplicates,
)

DATA_DIR = os.path.join(os.getcwd(), "..", "data")
REAL_GROUND_TRUTH_PATH = os.path.join(DATA_DIR, "raw", "Unilog-Sample_200_Items-Input-vs-Output.xlsx")
USE_REAL_DATA = os.path.exists(REAL_GROUND_TRUTH_PATH)
print("Real ground-truth file found -- using real data" if USE_REAL_DATA
      else "Real ground-truth file NOT found -- using mock Faucets fixture")

Real ground-truth file NOT found -- using mock Faucets fixture


## 2. Load data

Two paths:
- **Real data** (once Part 1 hands off `data/raw/*.xlsx` and their loaders in
  `part1_foundation`): loads the real 200-item Input/Delivery-Format sheets,
  the real `lov_lookup`, and the real `manufacturer_brand_lookup`.
- **Mock data** (fallback, used now): a hand-built 8-row Faucets fixture with
  a matching mini-LOV, so every downstream cell is runnable and testable
  before real files exist.

In [2]:
if USE_REAL_DATA:
    # --- Real data path -----------------------------------------------
    # Fill this in once part1_foundation exposes its loaders, e.g.:
    #
    # from unihack.part1_foundation.parsers import load_ground_truth, load_lov, load_manufacturer_brand_list
    # ground_truth_rows, true_classpaths, true_attributes = load_ground_truth(REAL_GROUND_TRUTH_PATH)
    # lov_lookup = load_lov(os.path.join(DATA_DIR, "raw", "FAUCETS_LOV.xlsx"))
    # manufacturer_brand_lookup = load_manufacturer_brand_list(
    #     os.path.join(DATA_DIR, "raw", "UniCat_Manufacturer_and_Brand_List.xlsx"))
    raise NotImplementedError("Wire up part1_foundation loaders here once available")

else:
    # --- Mock data path (current) --------------------------------------
    ALLOWED_CLASSPATHS = [
        "Plumbing > Faucets > Kitchen Faucets",
        "Plumbing > Faucets > Bathroom Sink Faucets",
        "Plumbing > Faucets > Bar & Prep Faucets",
    ]

    # attribute permitted values, keyed by classpath (mirrors FAUCETS_LOV.xlsx shape)
    LOV_BY_CLASSPATH = {
        "Plumbing > Faucets > Kitchen Faucets": {
            "Mount Type": ["Deck Mount", "Wall Mount"],
            "Finish": ["Chrome", "Stainless Steel", "Matte Black", "Oil Rubbed Bronze"],
            "Handle Type": ["Single Handle", "Double Handle"],
            "Spout Height": [],   # unit-bearing -- exempt from exact permitted-value match
        },
        "Plumbing > Faucets > Bathroom Sink Faucets": {
            "Mount Type": ["Deck Mount", "Wall Mount"],
            "Finish": ["Chrome", "Brushed Nickel", "Matte Black"],
            "Handle Type": ["Single Handle", "Double Handle", "Widespread"],
        },
        "Plumbing > Faucets > Bar & Prep Faucets": {
            "Mount Type": ["Deck Mount"],
            "Finish": ["Chrome", "Stainless Steel"],
            "Handle Type": ["Single Handle"],
        },
    }

    ground_truth_rows = [
        {"row_id": "f001", "part_desc": "PDSH-K100 Kitchen Faucet Single Handle Deck Mount Chrome 9.5in Spout",
         "mpn": "PDSH-K100", "manufacturer": "Frigidaire",
         "true_classpath": "Plumbing > Faucets > Kitchen Faucets",
         "true_attributes": {"Mount Type": "Deck Mount", "Finish": "Chrome", "Handle Type": "Single Handle", "Spout Height": "9.5 in"}},
        {"row_id": "f002", "part_desc": "PDSH-B200 Bathroom Widespread Faucet Brushed Nickel",
         "mpn": "PDSH-B200", "manufacturer": "Frigidaire",
         "true_classpath": "Plumbing > Faucets > Bathroom Sink Faucets",
         "true_attributes": {"Mount Type": "Deck Mount", "Finish": "Brushed Nickel", "Handle Type": "Widespread"}},
        {"row_id": "f003", "part_desc": "PDSH-P300 Prep Faucet Single Handle Stainless Deck Mount",
         "mpn": "PDSH-P300", "manufacturer": "Frigidaire",
         "true_classpath": "Plumbing > Faucets > Bar & Prep Faucets",
         "true_attributes": {"Mount Type": "Deck Mount", "Finish": "Stainless Steel", "Handle Type": "Single Handle"}},
        {"row_id": "f004", "part_desc": "PDSH-K101 Kitchen Faucet Double Handle Wall Mount Matte Black",
         "mpn": "PDSH-K101", "manufacturer": "Frigidaire",
         "true_classpath": "Plumbing > Faucets > Kitchen Faucets",
         "true_attributes": {"Mount Type": "Wall Mount", "Finish": "Matte Black", "Handle Type": "Double Handle"}},
        {"row_id": "f005", "part_desc": "PDSH-B201 Bath Sink Faucet Single Handle Chrome Deck Mount",
         "mpn": "PDSH-B201", "manufacturer": "Frigidaire",
         "true_classpath": "Plumbing > Faucets > Bathroom Sink Faucets",
         "true_attributes": {"Mount Type": "Deck Mount", "Finish": "Chrome", "Handle Type": "Single Handle"}},
    ]

    true_classpaths = {r["row_id"]: r["true_classpath"] for r in ground_truth_rows}
    true_attributes = {r["row_id"]: r["true_attributes"] for r in ground_truth_rows}

print(f"Loaded {len(ground_truth_rows)} ground-truth rows, {len(ALLOWED_CLASSPATHS)} allowed classpaths")

Loaded 5 ground-truth rows, 3 allowed classpaths


## 3. Model call

Stub `call_model` used during dev so this notebook runs with **no API key
and no cost**. Swap for a real Anthropic call when ready to test against
the live model (see commented block below).

In [3]:
def stub_call_model(system: str, user: str) -> str:
    """Deterministic fake model for dev iteration -- looks up the row by
    MPN embedded in the prompt and returns its known-correct answer, with
    slight noise on one row to keep the accuracy metric honest (not 100%)."""
    is_attr_prompt = "ATTRIBUTES TO EXTRACT" in user

    for r in ground_truth_rows:
        if r["mpn"] not in user:
            continue

        if is_attr_prompt:
            attrs = [
                {"attribute": k, "value": v, "confidence": 0.85,
                 "needs_unit_normalization": k == "Spout Height",
                 "raw_text_span": v}
                for k, v in r["true_attributes"].items()
            ]
            return json.dumps({"attributes": attrs})
        else:
            # deliberately get f004 wrong to exercise the gap-analysis path
            classpath = "Plumbing > Faucets > Bathroom Sink Faucets" if r["row_id"] == "f004" else r["true_classpath"]
            confidence = 0.35 if r["row_id"] == "f004" else 0.9
            return json.dumps({
                "classpath": classpath, "confidence": confidence,
                "alternative_candidates": [r["true_classpath"]] if r["row_id"] == "f004" else [],
                "reasoning": "stub response for dev iteration",
            })

    return json.dumps({"classpath": None, "confidence": 0.0, "alternative_candidates": [], "reasoning": "no match"})

call_model = stub_call_model

# --- Real model call (uncomment once ANTHROPIC_API_KEY is set) ----------
# import anthropic
# client = anthropic.Anthropic()
#
# def real_call_model(system: str, user: str) -> str:
#     resp = client.messages.create(
#         model="claude-sonnet-4-6", max_tokens=500,
#         system=system, messages=[{"role": "user", "content": user}],
#     )
#     return resp.content[0].text
#
# call_model = real_call_model

## 4. Run classification (Phase B)

In [4]:
classification_input = [
    {"row_id": r["row_id"], "part_desc": r["part_desc"], "mpn": r["mpn"], "manufacturer": r["manufacturer"]}
    for r in ground_truth_rows
]

classification_results = classify_batch(classification_input, ALLOWED_CLASSPATHS, call_model)

for r in classification_results:
    flag = "  <- NEEDS REVIEW" if r.needs_review else ""
    print(f"{r.row_id}: {r.classpath}  (confidence={r.confidence:.2f}){flag}")

f001: Plumbing > Faucets > Kitchen Faucets  (confidence=0.90)
f002: Plumbing > Faucets > Bathroom Sink Faucets  (confidence=0.90)
f003: Plumbing > Faucets > Bar & Prep Faucets  (confidence=0.90)
f004: Plumbing > Faucets > Bathroom Sink Faucets  (confidence=0.35)  <- NEEDS REVIEW
f005: Plumbing > Faucets > Bathroom Sink Faucets  (confidence=0.90)


In [5]:
classification_scores = score_classification_accuracy(classification_results, true_classpaths)
print(json.dumps(classification_scores, indent=2))

{
  "accuracy": 0.8,
  "n_scored": 5,
  "n_correct": 4,
  "n_flagged_low_confidence": 1,
  "misses": [
    {
      "row_id": "f004",
      "predicted": "Plumbing > Faucets > Bathroom Sink Faucets",
      "true": "Plumbing > Faucets > Kitchen Faucets",
      "confidence": 0.35,
      "reasoning": "stub response for dev iteration"
    }
  ]
}


### Gap analysis

Per the spec: low-confidence / misclassified rows should be inspected and
logged, not swept under the rug. This is a strength signal for judges.

In [6]:
for miss in classification_scores["misses"]:
    print(f"[MISS] {miss['row_id']}: predicted={miss['predicted']!r} true={miss['true']!r} "
          f"confidence={miss['confidence']:.2f}")
    print(f"        reasoning: {miss['reasoning']}")

[MISS] f004: predicted='Plumbing > Faucets > Bathroom Sink Faucets' true='Plumbing > Faucets > Kitchen Faucets' confidence=0.35
        reasoning: stub response for dev iteration


## 5. Run attribute extraction (Phase D)

In [7]:
# Only extract attributes for rows that classified successfully --
# no point extracting against a classpath we don't trust.
attribute_input = [
    {"row_id": r.row_id, "classpath": r.classpath, "part_desc": next(
        row["part_desc"] for row in ground_truth_rows if row["row_id"] == r.row_id),
     "permitted_values": LOV_BY_CLASSPATH[r.classpath]}
    for r in classification_results if r.classpath is not None
]

extracted_attributes = extract_attributes_batch(attribute_input, call_model)

for row_id, attrs in extracted_attributes.items():
    print(f"{row_id}:")
    for a in attrs:
        unit_flag = " [needs unit norm -> Part 3]" if a.needs_unit_normalization else ""
        print(f"    {a.attribute} = {a.value}{unit_flag}")

f001:
    Mount Type = Deck Mount
    Finish = Chrome
    Handle Type = Single Handle
    Spout Height = 9.5 in [needs unit norm -> Part 3]
f002:
    Mount Type = Deck Mount
    Finish = Brushed Nickel
    Handle Type = Widespread
f003:
    Mount Type = Deck Mount
    Finish = Stainless Steel
    Handle Type = Single Handle
f004:
    Mount Type = Wall Mount
    Finish = Matte Black
    Handle Type = Double Handle
f005:
    Mount Type = Deck Mount
    Finish = Chrome
    Handle Type = Single Handle


In [8]:
attribute_scores = score_attribute_extraction(extracted_attributes, true_attributes)
print(json.dumps(attribute_scores, indent=2))

{
  "precision": 1.0,
  "recall": 1.0,
  "f1": 1.0,
  "true_positives": 16,
  "false_positives": 0,
  "false_negatives": 0
}


## 6. De-duplication (optional, Phase C)

In [9]:
dedup_input = [
    {"row_id": r["row_id"], "mpn": r["mpn"], "part_desc": r["part_desc"], "manufacturer_canonical": r["manufacturer"]}
    for r in ground_truth_rows
]
# add one near-duplicate to actually exercise the fuzzy-match path
dedup_input.append({"row_id": "f001b", "mpn": "PDSH-K100", "part_desc": "PDSH-K100 Kitchen Faucet Single Handle Deck Mount Chrome",
                     "manufacturer_canonical": "Frigidaire"})

duplicate_groups = find_duplicates(dedup_input)
for g in duplicate_groups:
    print(f"{g.match_reason} (similarity={g.similarity:.2f}): {g.row_ids} -> representative: {g.representative_row_id}")

same_mpn (similarity=1.00): ['f001', 'f001b'] -> representative: f001


## 7. Summary (feeds Part 4's evaluation slide)

In [10]:
summary = {
    "classification_accuracy": round(classification_scores["accuracy"], 3),
    "classification_rows_scored": classification_scores["n_scored"],
    "classification_flagged_low_confidence": classification_scores["n_flagged_low_confidence"],
    "attribute_precision": round(attribute_scores["precision"], 3),
    "attribute_recall": round(attribute_scores["recall"], 3),
    "attribute_f1": round(attribute_scores["f1"], 3),
    "duplicate_groups_found": len(duplicate_groups),
}
print(json.dumps(summary, indent=2))

# Uncomment to hand off to Part 4:
# with open(os.path.join(DATA_DIR, "processed", "part2_summary.json"), "w") as f:
#     json.dump(summary, f, indent=2)

{
  "classification_accuracy": 0.8,
  "classification_rows_scored": 5,
  "classification_flagged_low_confidence": 1,
  "attribute_precision": 1.0,
  "attribute_recall": 1.0,
  "attribute_f1": 1.0,
  "duplicate_groups_found": 1
}


## Next steps
- [ ] Swap mock fixture (Section 2) for real `part1_foundation` loaders once files land
- [ ] Swap `stub_call_model` for the real Anthropic call (Section 3) and re-run
- [ ] Run against the full 200-item ground truth, not just this 5-row sample
- [ ] Widen `ALLOWED_CLASSPATHS` / `LOV_BY_CLASSPATH` from the real Faucets LOV
- [ ] Export classified + attributed rows for Part 3 (needs_unit_normalization flags intact)